# Graph RAG — PDF Ingestion Pipeline

This notebook walks through building a **Knowledge Graph** from a PDF document.
Each step is a separate cell so you can run and inspect the output before moving on.

| Step | What happens |
|------|--------------|
| 1 | Imports & credentials loaded from `.env` |
| 2 | PDF is read and split into overlapping text chunks |
| 3 | LLM reads each chunk and extracts entities + relationships as JSON |
| 4 | Duplicate entities/relationships are removed across all chunks |
| 5 | Clean data is written into Neo4j Aura as nodes and edges |

## Step 1 — Imports & Configuration

- `fitz` (PyMuPDF) reads the PDF
- `RecursiveCharacterTextSplitter` from LangChain splits text into chunks
- `OpenAI` client points to **OpenRouter** so we can use any LLM via one API
- `GraphDatabase` is the Neo4j Python driver
- Credentials are loaded from `.env` — never hard-coded

In [1]:
import os, json, re
import fitz  # pymupdf
from dotenv import load_dotenv
from openai import OpenAI
from neo4j import GraphDatabase
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

NEO4J_URI          = os.environ["NEO4J_URI"]
NEO4J_USERNAME     = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD     = os.environ["NEO4J_PASSWORD"]
OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]

LLM_MODEL     = "openai/gpt-4o-mini"
PDF_FILE      = "Corporate_Policies_Dataset_Final.pdf"
CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 150

openai_client = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")

print("✅ Step 1 complete — imports loaded, credentials set")

✅ Step 1 complete — imports loaded, credentials set


## Step 2 — Load PDF & Chunk Text

The PDF is too large to send to an LLM in one go, so we:
1. Extract all text from every page using `fitz`
2. Split it into chunks of **1000 characters** with a **150-character overlap**

The overlap ensures that sentences cut at a boundary are still captured in the next chunk.

In [2]:
doc       = fitz.open(PDF_FILE)
full_text = "\n".join(page.get_text() for page in doc)
splitter  = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks    = splitter.split_text(full_text)

print(f"✅ Step 2 complete — {len(chunks)} chunks extracted from '{PDF_FILE}'")
print(f"\n   Preview of chunk 0:\n{chunks[0][:300]}...")

✅ Step 2 complete — 2 chunks extracted from 'Corporate_Policies_Dataset_Final.pdf'

   Preview of chunk 0:
Corporate Policy Compendium 2026
Internal Use Only | Confidential
7. Global Remote Work & Hybrid Operations Policy
1. Eligibility and Work Models
Work Model
Required On-Site
Presence
Equipment Stipend
Approval
Level
On-Site
5 days per week
$0
Direct Manager
Hybrid
3 days per week
$500 one-time
Direc...


## Step 3 — LLM Extracts Entities & Relationships from Each Chunk

For every chunk we send a structured prompt to the LLM asking it to return **only JSON** with:
- `entities` — typed nodes like `Policy`, `Department`, `Employee`, `Role`, etc.
- `relationships` — directed edges like `APPLIES_TO`, `GOVERNS`, `ENTITLES`

The function strips any markdown fences the LLM might add before parsing the JSON.
Chunks that fail to parse are skipped gracefully.

In [3]:
EXTRACTION_PROMPT = """
You are a knowledge graph extractor. Given the text below, extract entities and relationships.

Return ONLY valid JSON in this exact format (no explanation, no markdown):
{{
  "entities": [
    {{"label": "EntityType", "name": "EntityName", "properties": {{"key": "value"}}}}
  ],
  "relationships": [
    {{"from": "EntityName1", "type": "RELATIONSHIP_TYPE", "to": "EntityName2"}}
  ]
}}

Rules:
- Entity labels must be one of: Policy, Department, Employee, Role, Benefit, Leave, Conduct, Penalty
- Relationship types must be UPPERCASE_SNAKE_CASE (e.g. APPLIES_TO, GOVERNS, ENTITLES, REPORTS_TO)
- Only extract what is explicitly stated in the text
- If nothing meaningful can be extracted, return: {{"entities": [], "relationships": []}}

Text:
\"\"\"
{chunk}
\"\"\"
"""

def extract_graph_from_chunk(chunk: str, chunk_idx: int) -> dict:
    resp = openai_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": EXTRACTION_PROMPT.format(chunk=chunk)}],
        temperature=0,
    )
    raw = resp.choices[0].message.content.strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    try:
        data = json.loads(raw)
        print(f"  Chunk {chunk_idx+1:>3}: {len(data.get('entities',[]))} entities, {len(data.get('relationships',[]))} relationships")
        return data
    except json.JSONDecodeError:
        print(f"  Chunk {chunk_idx+1:>3}: ✗ JSON parse failed — skipping")
        return {"entities": [], "relationships": []}

print(f"Extracting from {len(chunks)} chunks...\n")
all_extractions = [extract_graph_from_chunk(chunk, i) for i, chunk in enumerate(chunks)]

total_e = sum(len(x.get("entities", [])) for x in all_extractions)
total_r = sum(len(x.get("relationships", [])) for x in all_extractions)
print(f"\n✅ Step 3 complete — raw totals: {total_e} entities, {total_r} relationships (before dedup)")

Extracting from 2 chunks...



  Chunk   1: 6 entities, 5 relationships


  Chunk   2: 5 entities, 4 relationships

✅ Step 3 complete — raw totals: 11 entities, 9 relationships (before dedup)


## Step 4 — Deduplicate Entities & Relationships

The same entity (e.g. `"HR Department"`) will appear in many chunks.
We deduplicate using:
- A **dict keyed by name** for entities — first occurrence wins
- A **set of `(from, type, to)` tuples** for relationships — exact duplicates are dropped

This gives us a clean, unique set ready to insert into the graph.

In [4]:
seen_entities = {}
seen_rels     = set()
relationships = []

for extraction in all_extractions:
    for e in extraction.get("entities", []):
        name = e.get("name", "").strip()
        if name and name not in seen_entities:
            seen_entities[name] = e
    for r in extraction.get("relationships", []):
        key = (r.get("from", ""), r.get("type", ""), r.get("to", ""))
        if all(key) and key not in seen_rels:
            seen_rels.add(key)
            relationships.append(r)

entities = list(seen_entities.values())

print(f"✅ Step 4 complete — deduplicated: {len(entities)} entities, {len(relationships)} relationships")

✅ Step 4 complete — deduplicated: 11 entities, 9 relationships


## Step 4b — Test Neo4j Connectivity

Run this before Step 5. If it prints ❌, your Aura instance is paused or deleted.
Go to [console.neo4j.io](https://console.neo4j.io) → Resume the instance, then re-run.

In [5]:
_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
try:
    _driver.verify_connectivity()
    print("✅ Neo4j connection OK — safe to run Step 5")
except Exception as e:
    print(f"❌ Cannot connect: {e}")
    print("   → Go to console.neo4j.io and Resume your instance")
finally:
    _driver.close()

✅ Neo4j connection OK — safe to run Step 5


## Step 5 — Insert into Neo4j Aura

Two helper functions handle the Cypher writes:
- `insert_entity` — uses `MERGE` so re-running won't create duplicates; sanitizes property key names
- `insert_relationship` — `MATCH`es both endpoints then `MERGE`s the edge

The session runs three sub-steps:
- **5a** — wipe the existing graph
- **5b** — insert all nodes
- **5c** — insert all edges (relationships whose endpoints are missing are skipped)

> Ensure `.env` has `NEO4J_URI=neo4j+s://...` — the `neo4j+ssc://` scheme will cause a `ServiceUnavailable` error.

In [6]:
def _safe_key(k: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_]", "_", k).strip("_") or "prop"

def insert_entity(tx, entity: dict):
    label = entity.get("label", "Entity").strip()
    name  = entity.get("name", "").strip()
    if not name:
        return
    safe_props = {_safe_key(k): v for k, v in entity.get("properties", {}).items()}
    set_clause = (", " + ", ".join(f"n.{k} = ${k}" for k in safe_props)) if safe_props else ""
    tx.run(f"MERGE (n:{label} {{name: $name}}) SET n.name = $name{set_clause}", name=name, **safe_props)

def insert_relationship(tx, rel: dict):
    from_name = rel.get("from", "").strip()
    to_name   = rel.get("to", "").strip()
    rel_type  = rel.get("type", "RELATED_TO").strip().upper().replace(" ", "_")
    if not from_name or not to_name:
        return
    tx.run(
        f"MATCH (a {{name: $from_name}}) MATCH (b {{name: $to_name}}) MERGE (a)-[:{rel_type}]->(b)",
        from_name=from_name, to_name=to_name,
    )

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
    print("  5a ✓ Cleared existing graph")

    for e in entities:
        session.execute_write(insert_entity, e)
    print(f"  5b ✓ Inserted {len(entities)} nodes")

    skipped = 0
    for r in relationships:
        try:
            session.execute_write(insert_relationship, r)
        except Exception:
            skipped += 1
    print(f"  5c ✓ Inserted relationships (skipped {skipped} with missing nodes)")

driver.close()
print(f"\n✅ Pipeline complete! Nodes: {len(entities)} | Edges: {len(relationships)}")

  5a ✓ Cleared existing graph


  5b ✓ Inserted 11 nodes


  5c ✓ Inserted relationships (skipped 0 with missing nodes)

✅ Pipeline complete! Nodes: 11 | Edges: 9
